# Lotto Data Collection and Preprocessing

This notebook loads the lotto history dataset and converts it into a clean, analysis-ready table.

The goal of this step is to keep the original Excel-based workflow as the stable default while retaining the automated web-based workflow as an in-progress feature. The notebook then standardizes the raw columns, cleans numeric and text fields, creates a few derived variables, and saves the processed result for use in later notebooks.

### 1. Import

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().resolve().parent))

import pandas as pd
from src.data.load_data import load_lotto_source
from src.data.preprocess import preprocess_lotto_data, save_processed_lotto
from src.data.validate_data import (
    validate_number_ranges,
    validate_processed_columns,
    validate_unique_main_numbers,
)
from src.config import AUTO_RAW_LOTTO_FILE, RAW_LOTTO_FILE, PROCESSED_LOTTO_FILE

### 2. Select the Data Source and Load Raw Data

This notebook supports multiple input modes.

- `excel`: stable default mode that loads the local Excel file.
- `auto`: requests-based automatic collection path. This feature is currently under implementation and environment-specific verification.
- `auto_browser`: browser-based automatic collection path. This feature is also currently under implementation and may be blocked depending on the environment.

For now, the recommended mode is still `excel`.

To switch modes, change only the `DATA_SOURCE_MODE` value in the next code cell.
Detailed comments are included there so the user can choose a mode by editing just a few lines.

### 3. Preprocess the Loaded Data

In [3]:
# ============================================================
# Data source selection
# ============================================================
# Choose how to load the lotto history.
#
# Option 1: "excel"
# - Uses the existing local Excel file.
# - Stable default and currently recommended workflow.
#
# Option 2: "auto"
# - Requests-based automatic collection path.
# - This feature is currently under implementation and environment-specific verification.
#
# Option 3: "auto_browser"
# - Browser-based automatic collection path.
# - This feature is also currently under implementation and may be blocked by the official site.
#
# Recommended for now: keep DATA_SOURCE_MODE = "excel".
DATA_SOURCE_MODE = "excel"

# This file path is used only when DATA_SOURCE_MODE == "excel".
# Replace it if you want to point to another local Excel file.
CUSTOM_RAW_FILE = RAW_LOTTO_FILE

# This path is used only when DATA_SOURCE_MODE is an automated mode.
# The automatically collected raw history will be saved here as CSV.
AUTO_SAVE_FILE = AUTO_RAW_LOTTO_FILE

# You can also limit the collection range in automated modes.
# Keep START_ROUND = 1 and END_ROUND = None to collect the full history
# from the first draw to the latest available draw.
START_ROUND = 1
END_ROUND = None

df_raw = load_lotto_source(
    source=DATA_SOURCE_MODE,
    file_path=CUSTOM_RAW_FILE,
    auto_save_path=AUTO_SAVE_FILE,
    start_round=START_ROUND,
    end_round=END_ROUND,
)

df_raw.head()

/usr/local/lib/python3.10/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,No,회차,당첨번호,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,보너스,순위,당첨게임수,1게임당 당첨금액
0,1,1214,10,15,19,27,30,33,14,1등,12 명,"2,431,577,188 원"
1,2,1213,5,11,25,27,36,38,2,1등,18 명,"1,740,011,646 원"
2,3,1212,5,8,25,31,41,44,45,1등,12 명,"2,654,089,032 원"
3,4,1211,23,26,27,35,38,40,10,1등,14 명,"2,370,956,036 원"
4,5,1210,1,7,9,17,27,38,31,1등,24 명,"1,102,298,407 원"


In [4]:
df_raw.columns.tolist(), df_raw.shape

(['No',
  '회차',
  '당첨번호',
  'Unnamed: 3',
  'Unnamed: 4',
  'Unnamed: 5',
  'Unnamed: 6',
  'Unnamed: 7',
  '보너스',
  '순위',
  '당첨게임수',
  '1게임당 당첨금액'],
 (1214, 12))

In [4]:
df_clean = preprocess_lotto_data(df_raw)
validate_processed_columns(df_clean)
validate_number_ranges(df_clean)
validate_unique_main_numbers(df_clean)
df_clean.head()

,row_id,round,n1,n2,n3,n4,n5,n6,bonus,rank_text,winner_count,prize_amount,numbers,sum_main,odd_count,even_count,low_count,high_count
0,1214,1,10,23,29,33,37,40,16,1등,0,0,"10,23,29,33,37,40",172,4,2,1,5
1,1213,2,9,13,21,25,32,42,2,1등,1,2002006800,"9,13,21,25,32,42",142,4,2,3,3
2,1212,3,11,16,19,21,27,31,30,1등,1,2000000000,"11,16,19,21,27,31",125,5,1,4,2
3,1211,4,14,27,30,31,40,42,2,1등,0,0,"14,27,30,31,40,42",184,2,4,1,5
4,1210,5,16,24,29,40,41,42,3,1등,0,0,"16,24,29,40,41,42",192,2,4,1,5


In [5]:
save_processed_lotto(df_clean, PROCESSED_LOTTO_FILE)
print(PROCESSED_LOTTO_FILE)

/workspace/data/processed/lotto_cleaned.csv


In [6]:
df_saved = pd.read_csv(PROCESSED_LOTTO_FILE)
df_saved.head()

,row_id,round,n1,n2,n3,n4,n5,n6,bonus,rank_text,winner_count,prize_amount,numbers,sum_main,odd_count,even_count,low_count,high_count
0,1214,1,10,23,29,33,37,40,16,1등,0,0,"10,23,29,33,37,40",172,4,2,1,5
1,1213,2,9,13,21,25,32,42,2,1등,1,2002006800,"9,13,21,25,32,42",142,4,2,3,3
2,1212,3,11,16,19,21,27,31,30,1등,1,2000000000,"11,16,19,21,27,31",125,5,1,4,2
3,1211,4,14,27,30,31,40,42,2,1등,0,0,"14,27,30,31,40,42",184,2,4,1,5
4,1210,5,16,24,29,40,41,42,3,1등,0,0,"16,24,29,40,41,42",192,2,4,1,5


### 4. Summary

This notebook keeps the Excel-based workflow as the stable default and retains the automated web-collection workflow as an in-progress feature. It then standardizes the raw columns, creates analysis-ready derived variables, and saves the cleaned result as a reusable CSV file.

At the end of this step, the project has a consistent processed dataset that can be used by the EDA, statistical testing, feature engineering, and modeling notebooks.